# 03 — Shear Validation Against Truth

DP0.2 is based on the DC2 simulation, which means we have **truth tables**
with the input shear and true galaxy properties. This lets us measure
the multiplicative and additive biases of the shape measurement pipeline.

This is the key deliverable: demonstrating that we can **recover the
input shear from the measured shapes**.

**Bias model:** $\hat{g}_i = (1 + m_i)\, g_i^{\text{true}} + c_i$

**Run this on the Rubin Science Platform (RSP).**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

from lsst.daf.butler import Butler

## 1. Load Measured Shapes and Truth Catalog

In [ ]:
butler = Butler('dp02', collections='2.2i/runs/DP0.2')

tract = 4226
# Load multiple patches for better statistics
patches = [16, 17, 18, 23, 24, 25, 30, 31, 32]

all_objects = []
for patch in patches:
    try:
        obj = butler.get('objectTable', tract=tract, patch=patch)
        obj['patch'] = patch
        all_objects.append(obj)
    except Exception as e:
        print(f"Patch {patch}: {e}")

import pandas as pd
objects = pd.concat(all_objects, ignore_index=True)
print(f"Total objects across {len(all_objects)} patches: {len(objects)}")

In [ ]:
# Load the truth match table
# DP0.2 provides a pre-matched truth catalog
all_truth = []
for patch in patches:
    try:
        truth = butler.get('truthSummary', tract=tract, patch=patch)
        all_truth.append(truth)
    except Exception:
        # Try alternative truth table names
        try:
            truth = butler.get('truth_summary', tract=tract, patch=patch)
            all_truth.append(truth)
        except Exception as e:
            print(f"Truth for patch {patch}: {e}")

if all_truth:
    truth_cat = pd.concat(all_truth, ignore_index=True)
    print(f"Truth catalog: {len(truth_cat)} entries")
    print(f"Truth columns: {list(truth_cat.columns)[:20]}")
else:
    print("No truth tables found via Butler.")
    print("Alternative: query the TAP service for truth_match table.")
    print("See Section 1b below.")

### 1b. Alternative: Query Truth via TAP Service

If the Butler doesn't have a pre-matched truth table, use the RSP
TAP service to query the DC2 truth tables directly.

In [ ]:
# TAP query for truth-matched catalog
# This works on the RSP Notebook Aspect
from lsst.rsp import get_tap_service

service = get_tap_service('tap')

# Query: match objects to truth within 0.5 arcsec
# Adapt the table names for your DP0.2 version
query = """
SELECT
    obj.objectId,
    obj.coord_ra AS ra_meas,
    obj.coord_dec AS dec_meas,
    obj.refExtendedness,
    obj.i_cModelFlux,
    obj.i_cModelFluxErr,
    truth.ra AS ra_true,
    truth.dec AS dec_true,
    truth.redshift AS z_true,
    truth.mag_i AS mag_i_true,
    truth.shear_1 AS g1_true,
    truth.shear_2 AS g2_true,
    truth.convergence AS kappa_true,
    truth.ellipticity_1_true AS e1_int,
    truth.ellipticity_2_true AS e2_int,
    truth.is_variable,
    truth.truth_type
FROM
    dp02_dc2_catalogs.Object AS obj
JOIN
    dp02_dc2_catalogs.MatchesTruth AS mt ON obj.objectId = mt.objectId
JOIN
    dp02_dc2_catalogs.TruthSummary AS truth ON mt.id_truth_type = truth.id_truth_type
WHERE
    obj.detect_isPrimary = 1
    AND obj.refExtendedness = 1
    AND obj.tract = 4226
    AND truth.truth_type = 1
    AND obj.i_cModelFlux / obj.i_cModelFluxErr > 10
"""

# NOTE: Table and column names may differ — adapt to your DP0.2 schema.
# Run: service.search("SELECT * FROM tap_schema.tables").to_table().to_pandas()
# to discover available tables.

print("Submitting TAP query...")
try:
    results = service.search(query)
    matched = results.to_table().to_pandas()
    print(f"Matched galaxies: {len(matched)}")
except Exception as e:
    print(f"TAP query failed: {e}")
    print("Adapt the query to your DP0.2 schema.")
    print("Check available tables with:")
    print("  service.search('SELECT table_name FROM tap_schema.tables').to_table()")

## 2. Match Measured Shapes to Truth Shear

In [ ]:
# Extract the columns we need
# Adapt column names to what your query returned

# Find the shape columns in the object table
e1_col = [c for c in objects.columns if 'e1' in c.lower() and 'hsm' in c.lower()
          and ('regauss' in c.lower() or 'Regauss' in c)]
e2_col = [c for c in objects.columns if 'e2' in c.lower() and 'hsm' in c.lower()
          and ('regauss' in c.lower() or 'Regauss' in c)]

if e1_col and e2_col:
    e1_col, e2_col = e1_col[0], e2_col[0]
    print(f"Shape columns: {e1_col}, {e2_col}")
else:
    print("HSM Regauss columns not found. Available shape columns:")
    print([c for c in objects.columns if 'e1' in c.lower() or 'e2' in c.lower()])

In [ ]:
# If using the TAP-matched catalog:
# (If using Butler truth tables, adapt the column names)

try:
    g1_true = matched['g1_true'].values
    g2_true = matched['g2_true'].values
    z_true = matched['z_true'].values

    # Get measured shapes for the matched objects
    # (This requires joining on objectId — adapt to your data)
    # For now, assume matched has objectId that we can join

    print(f"True shear range: g1 = [{g1_true.min():.4f}, {g1_true.max():.4f}]")
    print(f"                  g2 = [{g2_true.min():.4f}, {g2_true.max():.4f}]")
    print(f"Redshift range: z = [{z_true.min():.2f}, {z_true.max():.2f}]")
except NameError:
    print("No matched catalog available yet.")
    print("This cell will work once the TAP query or Butler truth tables are loaded.")
    print("\nFor now, we'll demonstrate the analysis with synthetic data below.")

## 3. Measure Multiplicative and Additive Bias

The key measurement: fit the relation between measured and true shear.

$$\langle e_i \rangle = (1 + m_i) \, g_i^{\text{true}} + c_i$$

We bin galaxies by their true shear, compute the mean measured
ellipticity in each bin, and fit a line.

In [ ]:
def measure_bias(g_true, e_meas, n_bins=15):
    """
    Estimate multiplicative (m) and additive (c) bias.

    Bins galaxies by true shear, computes mean measured e in each bin,
    and fits: <e_meas> = (1+m) * g_true + c

    Returns: m, c, m_err, c_err, bin_centers, bin_means, bin_errs
    """
    # Remove outliers
    good = np.isfinite(g_true) & np.isfinite(e_meas) & (np.abs(e_meas) < 2)
    g_true = g_true[good]
    e_meas = e_meas[good]

    # Bin by true shear
    bin_edges = np.linspace(g_true.min(), g_true.max(), n_bins + 1)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    bin_means = np.zeros(n_bins)
    bin_errs = np.zeros(n_bins)

    for i in range(n_bins):
        mask = (g_true >= bin_edges[i]) & (g_true < bin_edges[i+1])
        if mask.sum() > 5:
            bin_means[i] = np.mean(e_meas[mask])
            bin_errs[i] = np.std(e_meas[mask]) / np.sqrt(mask.sum())
        else:
            bin_means[i] = np.nan
            bin_errs[i] = np.nan

    # Fit: e = (1+m)*g + c
    valid = np.isfinite(bin_means)
    if valid.sum() > 2:
        slope, intercept, r, p, se = stats.linregress(
            bin_centers[valid], bin_means[valid]
        )
        m = slope - 1.0  # multiplicative bias
        c = intercept     # additive bias
        m_err = se
        c_err = se * np.sqrt(np.mean(bin_centers[valid]**2))
    else:
        m, c, m_err, c_err = np.nan, np.nan, np.nan, np.nan

    return m, c, m_err, c_err, bin_centers, bin_means, bin_errs

In [ ]:
# --- If real matched data is available, use it ---
# --- Otherwise, demonstrate with synthetic data ---

try:
    # Use real data
    e1_meas = matched['e1_meas'].values  # adapt column name
    e2_meas = matched['e2_meas'].values
    use_synthetic = False
except (NameError, KeyError):
    # Synthetic demonstration
    print("Using synthetic data for demonstration.")
    print("Replace with real matched data when available.\n")

    rng = np.random.default_rng(42)
    n_gal = 50000

    # True shear field (varies spatially in DC2)
    g1_true = rng.uniform(-0.05, 0.05, n_gal)
    g2_true = rng.uniform(-0.05, 0.05, n_gal)

    # Intrinsic shapes (shape noise)
    e_int = rng.rayleigh(0.25, n_gal)
    phi_int = rng.uniform(0, np.pi, n_gal)
    e1_int = e_int * np.cos(2 * phi_int)
    e2_int = e_int * np.sin(2 * phi_int)

    # Simulated measurement: introduce a small m and c bias
    m_input = -0.02   # 2% multiplicative bias
    c1_input = 0.0003  # small additive
    c2_input = -0.0001

    e1_meas = (1 + m_input) * (g1_true + e1_int) + c1_input + rng.normal(0, 0.01, n_gal)
    e2_meas = (1 + m_input) * (g2_true + e2_int) + c2_input + rng.normal(0, 0.01, n_gal)

    z_true = rng.uniform(0.2, 2.0, n_gal)
    use_synthetic = True

In [ ]:
# Measure bias
m1, c1, m1_err, c1_err, bins1, means1, errs1 = measure_bias(g1_true, e1_meas)
m2, c2, m2_err, c2_err, bins2, means2, errs2 = measure_bias(g2_true, e2_meas)

print("=" * 50)
print("SHEAR CALIBRATION RESULTS")
print("=" * 50)
print(f"m1 = {m1:.4f} ± {m1_err:.4f}")
print(f"c1 = {c1:.6f} ± {c1_err:.6f}")
print(f"m2 = {m2:.4f} ± {m2_err:.4f}")
print(f"c2 = {c2:.6f} ± {c2_err:.6f}")
print()
print("LSST requirements: |m| < 0.003, |c| < 0.0003")
print(f"m1 passes: {abs(m1) < 0.003}")
print(f"c1 passes: {abs(c1) < 0.0003}")
if use_synthetic:
    print(f"\n(Synthetic input: m={m_input}, c1={c1_input}, c2={c2_input})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# g1 component
ax = axes[0]
valid = np.isfinite(means1)
ax.errorbar(bins1[valid], means1[valid], yerr=errs1[valid],
            fmt='o', color='steelblue', ms=6, capsize=3, label='Data')
g_plot = np.linspace(bins1[valid].min(), bins1[valid].max(), 100)
ax.plot(g_plot, (1 + m1) * g_plot + c1, 'r-', lw=2,
        label=f'Fit: m={m1:.4f}, c={c1:.6f}')
ax.plot(g_plot, g_plot, 'k--', lw=1, alpha=0.5, label='Ideal: m=0, c=0')
ax.set_xlabel('$g_1^{\\rm true}$', fontsize=13)
ax.set_ylabel('$\\langle e_1 \\rangle$', fontsize=13)
ax.set_title('$g_1$ shear calibration', fontsize=14)
ax.legend(fontsize=10)

# g2 component
ax = axes[1]
valid = np.isfinite(means2)
ax.errorbar(bins2[valid], means2[valid], yerr=errs2[valid],
            fmt='o', color='forestgreen', ms=6, capsize=3, label='Data')
ax.plot(g_plot, (1 + m2) * g_plot + c2, 'r-', lw=2,
        label=f'Fit: m={m2:.4f}, c={c2:.6f}')
ax.plot(g_plot, g_plot, 'k--', lw=1, alpha=0.5, label='Ideal: m=0, c=0')
ax.set_xlabel('$g_2^{\\rm true}$', fontsize=13)
ax.set_ylabel('$\\langle e_2 \\rangle$', fontsize=13)
ax.set_title('$g_2$ shear calibration', fontsize=14)
ax.legend(fontsize=10)

plt.suptitle('Shear Bias Measurement: $\\langle e \\rangle = (1+m)\\,g + c$',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../../figures/shear_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Bias as a Function of Galaxy Properties

Shear bias can depend on galaxy SNR, size, redshift, and morphology.
Understanding these dependencies is critical for calibration.

In [ ]:
# Measure m as a function of redshift
z_bins = np.array([0.2, 0.5, 0.8, 1.1, 1.5, 2.0])
m_vs_z = []
m_err_vs_z = []
z_centers = 0.5 * (z_bins[:-1] + z_bins[1:])

for i in range(len(z_bins) - 1):
    zmask = (z_true >= z_bins[i]) & (z_true < z_bins[i+1])
    if zmask.sum() > 100:
        m_z, c_z, me_z, ce_z, _, _, _ = measure_bias(
            g1_true[zmask], e1_meas[zmask], n_bins=10
        )
        m_vs_z.append(m_z)
        m_err_vs_z.append(me_z)
    else:
        m_vs_z.append(np.nan)
        m_err_vs_z.append(np.nan)

m_vs_z = np.array(m_vs_z)
m_err_vs_z = np.array(m_err_vs_z)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

ax.errorbar(z_centers, m_vs_z, yerr=m_err_vs_z,
            fmt='o-', color='steelblue', ms=8, capsize=4, lw=2)
ax.axhline(0, color='black', ls='-', lw=0.8)
ax.axhspan(-0.003, 0.003, color='green', alpha=0.15, label='LSST requirement')

ax.set_xlabel('Redshift $z$', fontsize=13)
ax.set_ylabel('Multiplicative bias $m$', fontsize=13)
ax.set_title('Shear bias vs. redshift (tomographic bins)', fontsize=14)
ax.legend(fontsize=11)
ax.set_xlim(0, 2.2)

plt.tight_layout()
plt.savefig('../../figures/m_vs_redshift.png', dpi=150, bbox_inches='tight')
plt.show()

print("If m varies with z, the shear calibration must be applied")
print("per tomographic bin — a flat correction is insufficient.")

## 5. Shear Map

Visualize the measured shear field as a whisker plot and compare
to the true shear from the DC2 truth tables.

In [ ]:
# Create a binned shear map
# Use RA/Dec (or x/y for a single patch)

try:
    ra = matched['ra_meas'].values
    dec = matched['dec_meas'].values
except (NameError, KeyError):
    # Synthetic positions
    ra = rng.uniform(55.0, 56.0, n_gal)
    dec = rng.uniform(-28.5, -27.5, n_gal)

# Bin into a grid
nx_map, ny_map = 12, 12
ra_edges = np.linspace(ra.min(), ra.max(), nx_map + 1)
dec_edges = np.linspace(dec.min(), dec.max(), ny_map + 1)

g1_map_meas = np.zeros((ny_map, nx_map))
g2_map_meas = np.zeros((ny_map, nx_map))
g1_map_true = np.zeros((ny_map, nx_map))
g2_map_true = np.zeros((ny_map, nx_map))
count_map = np.zeros((ny_map, nx_map))

for iy in range(ny_map):
    for ix in range(nx_map):
        mask = ((ra >= ra_edges[ix]) & (ra < ra_edges[ix+1]) &
                (dec >= dec_edges[iy]) & (dec < dec_edges[iy+1]))
        if mask.sum() > 10:
            g1_map_meas[iy, ix] = np.mean(e1_meas[mask])
            g2_map_meas[iy, ix] = np.mean(e2_meas[mask])
            g1_map_true[iy, ix] = np.mean(g1_true[mask])
            g2_map_true[iy, ix] = np.mean(g2_true[mask])
            count_map[iy, ix] = mask.sum()

ra_centers = 0.5 * (ra_edges[:-1] + ra_edges[1:])
dec_centers = 0.5 * (dec_edges[:-1] + dec_edges[1:])
ra_grid, dec_grid = np.meshgrid(ra_centers, dec_centers)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

scale = 0.5
quiver_kw = dict(scale=scale, headwidth=0, headlength=0,
                 headaxislength=0, linewidth=1.5, pivot='mid')

# True shear
ax = axes[0]
ax.quiver(ra_grid, dec_grid, g1_map_true, g2_map_true,
          color='red', **quiver_kw)
ax.set_xlabel('RA (deg)'); ax.set_ylabel('Dec (deg)')
ax.set_title('True shear field', fontsize=13)
ax.set_aspect('equal')

# Measured shear
ax = axes[1]
ax.quiver(ra_grid, dec_grid, g1_map_meas, g2_map_meas,
          color='steelblue', **quiver_kw)
ax.set_xlabel('RA (deg)'); ax.set_ylabel('Dec (deg)')
ax.set_title('Measured shear (mean $e$)', fontsize=13)
ax.set_aspect('equal')

# Galaxy density (number per bin)
ax = axes[2]
im = ax.pcolormesh(ra_edges, dec_edges, count_map, cmap='viridis')
plt.colorbar(im, ax=ax, label='Galaxies per bin')
ax.set_xlabel('RA (deg)'); ax.set_ylabel('Dec (deg)')
ax.set_title('Galaxy number density', fontsize=13)
ax.set_aspect('equal')

plt.suptitle('Shear Maps: True vs. Measured', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../../figures/shear_maps.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

This notebook demonstrates the complete shear validation pipeline:

1. **Load** measured shapes and truth catalogs from DP0.2
2. **Match** measured objects to truth via TAP or pre-matched tables
3. **Measure** multiplicative (m) and additive (c) bias
4. **Characterize** bias dependence on galaxy properties (z, SNR)
5. **Visualize** the shear field and compare to truth

### Key results to report:
- m and c values (with uncertainties)
- Whether they meet LSST requirements (|m| < 0.003, |c| < 0.0003)
- Redshift dependence of the bias
- Qualitative agreement of shear maps

**Next:** [04_colours_and_photoz.ipynb](04_colours_and_photoz.ipynb) — Galaxy colours and photo-z from the pipeline